In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/avazu-ctr-prediction/sampleSubmission.gz
/kaggle/input/competitions/avazu-ctr-prediction/train.gz
/kaggle/input/competitions/avazu-ctr-prediction/test.gz


In [2]:
# ===========================
# Avazu Hybrid CTR (DIN + SASRec-style Transformer)
# - Current context (tabular categorical features) + History sequence (device_id)
# - History encoded by causal Transformer (SASRec-style)
# - DIN attention: query=current, keys/values=history
# - Metrics: LogLoss + AUC
# - Leakage-safe split by target hour (time split)
# - Memory-safe dataset: store only target row indices
# ===========================

import gc
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import log_loss, roc_auc_score

# ---------------------------
# Config
# ---------------------------
INPUT_DIR = Path("/kaggle/input")
AVAZU_COMPETITION = "avazu-ctr-prediction"
AVAZU_TRAIN_FILE = Path(AVAZU_COMPETITION) / "train.gz"
AVAZU_TRAIN_PATHS = (
    INPUT_DIR / "competitions" / AVAZU_TRAIN_FILE,
    INPUT_DIR / AVAZU_TRAIN_FILE,
)
NROWS = 2_000_000

USER_COL_CANDIDATES = ["device_id", "device_ip"]  # prefer device_id else device_ip
K_EVENTS = 20

# Use strong categorical set (safe + common)
TOP_FEATURES = [
    "C1", "banner_pos",
    "site_id", "site_domain", "site_category",
    "app_id", "app_domain", "app_category",
    "device_type", "device_conn_type",
    "C14", "C15", "C16", "C17", "C18", "C19", "C20", "C21"
]

EMBED_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.2

BATCH_SIZE = 512
EPOCHS = 8
PATIENCE = 2
LR = 1e-3
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------
# Load
# ---------------------------
DATA_PATH = next((path for path in AVAZU_TRAIN_PATHS if path.is_file()), None)
if DATA_PATH is None:
    print("Connecting the official Avazu competition data with kagglehub...", flush=True)
    try:
        import kagglehub

        DATA_PATH = Path(kagglehub.competition_download(
            AVAZU_COMPETITION,
            path=AVAZU_TRAIN_FILE.name,
        ))
    except Exception as exc:
        raise RuntimeError(
            "Avazu 대회 데이터 자동 연결에 실패했습니다.\n"
            "Kaggle 오른쪽 Input → Add Input에서 Click-Through Rate Prediction 대회를 추가하세요.\n"
            "규칙 동의가 필요하면 공식 대회 데이터 페이지에서 안내를 확인하세요.\n"
            "https://www.kaggle.com/competitions/avazu-ctr-prediction/data?select=train.gz"
        ) from exc

print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH, compression="gzip", nrows=NROWS)
df["hour"] = df["hour"].astype(np.int64)

USER_COL = None
for c in USER_COL_CANDIDATES:
    if c in df.columns:
        USER_COL = c
        break
if USER_COL is None:
    raise RuntimeError("No user column found among: " + str(USER_COL_CANDIDATES))

print("Using user key:", USER_COL)
print("Shape:", df.shape)
print("click ratio:", df["click"].mean())

# choose feature columns that exist
feat_cols = [c for c in TOP_FEATURES if c in df.columns and c not in ["id", "click", "hour"]]
print("Using #feature-cols:", len(feat_cols))
print("Feature cols:", feat_cols)

# ---------------------------
# Sort by (user, hour) to make per-user rows contiguous and time-ordered
# ---------------------------
df[USER_COL] = df[USER_COL].astype(str)
df = df.sort_values([USER_COL, "hour"]).reset_index(drop=True)

# factorize user to integer codes (now contiguous blocks)
user_codes, user_uniques = pd.factorize(df[USER_COL], sort=False)
user_codes = user_codes.astype(np.int32)

# boundaries of each user block
# starts: indices where user changes, plus 0
# ends: next start, plus N
diff = np.diff(user_codes, prepend=user_codes[0])
starts = np.where(diff != 0)[0]
starts = np.concatenate(([0], starts))
ends = np.concatenate((starts[1:], [len(df)]))

print("Unique users:", len(starts))

# ---------------------------
# Tokenize features with factorize per column into one shared vocab namespace
# token ids:
#   0 = PAD
#   1 = reserved
#   tokens start at 2, with per-column offsets
# ---------------------------
print("\nFactorizing feature columns into global token ids...")
offset = 2
token_cols = []
for c in feat_cols:
    codes, uniques = pd.factorize(df[c].astype(str), sort=False)
    codes = codes.astype(np.int32)
    tok = codes + offset
    token_cols.append(tok)
    offset += (len(uniques) + 1)
    print(f"  {c:<14} uniques={len(uniques):,} offset_now={offset:,}")

VOCAB_SIZE = offset + 1
F = len(token_cols)
print("\nF(tokens per event):", F)
print("VOCAB_SIZE:", VOCAB_SIZE)

event_tokens = np.stack(token_cols, axis=1).astype(np.int32)  # (N, F)
clicks = df["click"].astype(np.int8).values                   # (N,)
hours  = df["hour"].astype(np.int64).values                   # (N,)

# free df
del df, token_cols
gc.collect()

# ---------------------------
# Build sample index: store ONLY target row indices
# For each user block [s:e), valid target rows are s+K ... e-1
# ---------------------------
print("\nBuilding sample index (target row indices)...")
target_rows = []
for s, e in zip(starts, ends):
    m = e - s
    if m <= K_EVENTS:
        continue
    # targets at positions K..m-1 (predict click at target, using previous K events)
    target_rows.append(np.arange(s + K_EVENTS, e, dtype=np.int64))

target_rows = np.concatenate(target_rows, axis=0)
print("Total samples:", len(target_rows))
print("Positive ratio:", float(clicks[target_rows].mean()))

# ---------------------------
# Time-based split by target hour (leakage-safe)
# ---------------------------
target_hours = hours[target_rows]
q1 = np.quantile(target_hours, 0.80)
q2 = np.quantile(target_hours, 0.90)

train_idx = np.where(target_hours <= q1)[0]
val_idx   = np.where((target_hours > q1) & (target_hours <= q2))[0]
test_idx  = np.where(target_hours > q2)[0]

print(f"\nSplit sizes train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")
print("Train click mean:", float(clicks[target_rows[train_idx]].mean()))
print("Val   click mean:", float(clicks[target_rows[val_idx]].mean()))
print("Test  click mean:", float(clicks[target_rows[test_idx]].mean()))

# =========================
# LABEL SHUFFLE SANITY CHECK (train only)
# =========================
import numpy as np

DO_LABEL_SHUFFLE = False

if DO_LABEL_SHUFFLE:
    rng = np.random.default_rng(2026)

    # train에 해당하는 '실제 row index t'들
    t_train = target_rows[train_idx]

    # 원본 clicks 보호(선택): 복사해서 사용
    clicks = clicks.copy()

    # train 라벨만 뽑아서 섞은 뒤 다시 넣기
    y = clicks[t_train].copy()
    rng.shuffle(y)
    clicks[t_train] = y

    print(f"✅ Label shuffle enabled: shuffled clicks at {len(t_train):,} train target rows")

# ---------------------------
# Dataset (memory-safe)
# Returns:
#   hist_tokens: (K, F)  from rows [t-K .. t-1]
#   cur_tokens:  (F,)    from row [t]
#   y:           float   click at row t
# ---------------------------
class AvazuHybridDataset(Dataset):
    def __init__(self, target_rows_all, subset_indices, event_tokens, clicks, K):
        self.target_rows_all = target_rows_all
        self.subset_indices = subset_indices
        self.event_tokens = event_tokens
        self.clicks = clicks
        self.K = K

    def __len__(self):
        return len(self.subset_indices)

    def __getitem__(self, i):
        t = self.target_rows_all[self.subset_indices[i]]
        hist = self.event_tokens[t - self.K : t]    # (K, F)
        cur  = self.event_tokens[t]                 # (F,)
        y = float(self.clicks[t])
        return torch.tensor(hist, dtype=torch.long), torch.tensor(cur, dtype=torch.long), torch.tensor(y, dtype=torch.float32)

def make_loader(subset_idx, shuffle):
    ds = AvazuHybridDataset(target_rows, subset_idx, event_tokens, clicks, K_EVENTS)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False, num_workers=0)

train_loader = make_loader(train_idx, shuffle=True)
val_loader   = make_loader(val_idx, shuffle=False)
test_loader  = make_loader(test_idx, shuffle=False)

# ---------------------------
# Model: (Current encoder) + (History Transformer) + (DIN attention) -> CTR head
# - event embedding = mean of feature token embeddings
# - history transformer over K event embeddings (causal)
# - DIN attention: query=current, keys=history
# ---------------------------
class HybridDINSASRecCTR(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, K, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.K = K
        self.embed_dim = embed_dim

        enc = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=num_layers)

        # CTR head (query + context + interaction)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 4, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1)
        )

    def _event_repr(self, tok2d):
        # tok2d: (..., F)
        e = self.emb(tok2d)            # (..., F, D)
        e = self.dropout(e)
        return e.mean(dim=-2)          # (..., D)

    def _causal_mask(self, L, device):
        m = torch.triu(torch.ones(L, L, device=device), diagonal=1)
        return m.masked_fill(m == 1, float("-inf"))

    def forward(self, hist_tokens, cur_tokens):
        # hist_tokens: (B, K, F)
        # cur_tokens:  (B, F)
        B, K, F = hist_tokens.shape

        # event-level embeddings
        hist_e = self._event_repr(hist_tokens)      # (B, K, D)
        cur_e  = self._event_repr(cur_tokens)       # (B, D)

        # history encoding (SASRec-style causal)
        mask = self._causal_mask(K, hist_tokens.device)
        H = self.encoder(hist_e, mask=mask)         # (B, K, D)

        # DIN attention: query=cur_e, keys/values=H
        # score = dot(q, H_t) / sqrt(D)
        q = cur_e.unsqueeze(1)                      # (B, 1, D)
        scores = (H * q).sum(dim=-1) / (self.embed_dim ** 0.5)   # (B, K)
        attn = torch.softmax(scores, dim=-1)        # (B, K)
        context = (H * attn.unsqueeze(-1)).sum(dim=1)            # (B, D)

        # head features
        x = torch.cat([cur_e, context, cur_e * context, cur_e - context], dim=-1)  # (B, 4D)
        logit = self.mlp(x).squeeze(1)              # (B,)
        return logit

model = HybridDINSASRecCTR(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    K=K_EVENTS,
    dropout=DROPOUT
).to(DEVICE)

print(f"\nModel on {DEVICE}, params={sum(p.numel() for p in model.parameters()):,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ---------------------------
# Eval: LogLoss + AUC
# ---------------------------
@torch.no_grad()
def eval_logloss_auc(model, loader):
    model.eval()
    ys_, ps_ = [], []
    for hist, cur, y in loader:
        hist = hist.to(DEVICE)
        cur  = cur.to(DEVICE)
        y    = y.to(DEVICE)

        logits = model(hist, cur)
        probs = torch.sigmoid(logits)

        ys_.append(y.detach().cpu().numpy())
        ps_.append(probs.detach().cpu().numpy())

    y = np.concatenate(ys_, axis=0)
    p = np.concatenate(ps_, axis=0)
    p = np.clip(p, 1e-7, 1 - 1e-7)

    ll = log_loss(y, p)
    auc = roc_auc_score(y, p) if len(np.unique(y)) >= 2 else 0.0
    return ll, auc

# ---------------------------
# Train with early stopping (by Val LogLoss)
# ---------------------------
best_val_ll = float("inf")
pat = 0

print("\nStarting training...\n")
for epoch in range(1, EPOCHS + 1):
    model.train()
    for hist, cur, y in train_loader:
        hist = hist.to(DEVICE)
        cur  = cur.to(DEVICE)
        y    = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(hist, cur)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()

    val_ll, val_auc = eval_logloss_auc(model, val_loader)
    print(f"Epoch {epoch:02d} | Val LogLoss {val_ll:.5f} | Val AUC {val_auc:.5f}")

    if val_ll < best_val_ll:
        best_val_ll = val_ll
        pat = 0
        torch.save({"model_state": model.state_dict()}, "best_hybrid_din_sasrec.pt")
        print("  → Saved best model (by Val LogLoss)")
    else:
        pat += 1
        if pat >= PATIENCE:
            print(f"Early stopping (patience={PATIENCE}) at epoch {epoch}")
            break

# ---------------------------
# Test
# ---------------------------
ckpt = torch.load("best_hybrid_din_sasrec.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])

test_ll, test_auc = eval_logloss_auc(model, test_loader)

print("\n" + "="*60)
print("FINAL TEST SET EVALUATION (Hybrid DIN + SASRec-style)")
print("="*60)
print(f"Best Val LogLoss: {best_val_ll:.5f}")
print(f"Test LogLoss:     {test_ll:.5f}")
print(f"Test AUC:         {test_auc:.5f}")
print("="*60)

CUDA available: True
GPU: Tesla T4
Loading: /kaggle/input/competitions/avazu-ctr-prediction/train.gz
Using user key: device_id
Shape: (2000000, 24)
click ratio: 0.1616045
Using #feature-cols: 18
Feature cols: ['C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21']
Unique users: 163262

Factorizing feature columns into global token ids...
  C1             uniques=7 offset_now=10
  banner_pos     uniques=7 offset_now=18
  site_id        uniques=2,473 offset_now=2,492
  site_domain    uniques=2,653 offset_now=5,146
  site_category  uniques=22 offset_now=5,169
  app_id         uniques=3,108 offset_now=8,278
  app_domain     uniques=206 offset_now=8,485
  app_category   uniques=27 offset_now=8,513
  device_type    uniques=4 offset_now=8,518
  device_conn_type uniques=4 offset_now=8,523
  C14            uniques=689 offset_now=9,213
  C15            unique

/tmp/ipykernel_49/684129102.py:273: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc, num_layers=num_layers)



Model on cuda, params=734,977

Starting training...

Epoch 01 | Val LogLoss 0.40364 | Val AUC 0.74218
  → Saved best model (by Val LogLoss)
Epoch 02 | Val LogLoss 0.40168 | Val AUC 0.74576
  → Saved best model (by Val LogLoss)
Epoch 03 | Val LogLoss 0.40079 | Val AUC 0.74744
  → Saved best model (by Val LogLoss)
Epoch 04 | Val LogLoss 0.40098 | Val AUC 0.74648
Epoch 05 | Val LogLoss 0.40087 | Val AUC 0.74804
Early stopping (patience=2) at epoch 5

FINAL TEST SET EVALUATION (Hybrid DIN + SASRec-style)
Best Val LogLoss: 0.40079
Test LogLoss:     0.40010
Test AUC:         0.72656
